# BenoStreamDB: 15-Minute Interactive Tour (Part 1)
## Installation, SciFact Ingestion & Hybrid Search

This notebook introduces **BenoStreamDB** — a serverless index-streaming vector data lake built in Rust with high-performance Python bindings.

In this demo, we:
1. Load real scientific peer-reviewed papers from Hugging Face (**SciFact** benchmark corpus).
2. Embed the abstracts using a local sentence-transformer (`all-MiniLM-L6-v2`).
3. Store and index the dataset with **Overlay Indexing** (HNSW vector index + Inverted index).
4. Perform **scalar filtering**, **dense vector search**, and **hybrid search**.

### 1. Environment Setup & Hardware Acceleration

Import libraries and detect hardware compute contexts (NVIDIA CUDA, Intel GPU, or CPU).

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import shutil
import pandas as pd
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import benostreamdb as bsdb

# Pandas formatting
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 1000)

# Hardware compute context
ctx = None
for target in ["cuda", "intel"]:
    try:
        ctx = bsdb.ComputeContext(target)
        print(f"Hardware Acceleration: {target.upper()} GPU active")
        break
    except Exception:
        pass
if ctx is None:
    print("Hardware Acceleration: CPU active")

Hardware Acceleration: CUDA GPU active


### 2. Download Real Scientific Dataset from Hugging Face

We use the **SciFact** corpus (`mteb/scifact`), containing scientific papers with research titles and full abstracts.

In [2]:
print("Downloading SciFact corpus from Hugging Face...")
try:
    corpus = load_dataset("mteb/scifact", "corpus", split="corpus[:200]")
except Exception:
    corpus = load_dataset("BeIR/scifact", "corpus", split="corpus[:200]")

df = pd.DataFrame(corpus).rename(columns={"_id": "doc_id"})
print(f"Loaded {len(df)} scientific papers.")
df[["doc_id", "title", "text"]].head()

Loaded 200 scientific papers.


,doc_id,title,text
0,4983,Microstructural development of human newborn cerebral white matter assessed in vivo by...,Alterations of the architecture of cerebral white matter in the developing human brain...
1,5836,Induction of myelodysplasia by myeloid-derived suppressor cells.,Myelodysplastic syndromes (MDS) are age-dependent stem cell malignancies that share bi...
2,7912,"BC1 RNA, the transcript from a master gene for ID element amplification, is able to pr...",ID elements are short interspersed elements (SINEs) found in high copy number in many ...
3,18670,The DNA Methylome of Human Peripheral Blood Mononuclear Cells,DNA methylation plays an important role in biological processes in human health and di...
4,19238,The human myelin basic protein gene is included within a 179-kilobase transcription un...,Two human Golli (for gene expressed in the oligodendrocyte lineage)-MBP (for myelin ba...


### 3. Embed & Ingest into BenoStreamDB

We embed the abstracts with `all-MiniLM-L6-v2` and write them to a BenoStreamDB table.
BenoStreamDB writes standard Apache Parquet with sidecar overlay indexes (HNSW for vectors, Inverted index for text/identifiers).

In [3]:
# Clean previous table directory
db_path = "scifact_news_db"
if os.path.exists(db_path):
    shutil.rmtree(db_path)

print("Embedding scientific abstracts...")
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["text"].tolist())
df["embedding"] = [list(e) for e in embeddings]

print("Ingesting into BenoStreamDB table...")
table = bsdb.Table(db_path, device=ctx)

# Add indexes: HNSW vector index on 'embedding', Inverted index on 'doc_id' and 'title'
table.add_index_columns(["embedding", "doc_id", "title"])

table.write_pandas(df)
table.commit()
print("Table successfully written and committed!")

Embedding scientific abstracts...


Loading weights:   0%|                                 | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 8485.49it/s]

Ingesting into BenoStreamDB table...
Table successfully written and committed!


In [4]:
# Inspect committed table
table_df = table.to_pandas()
print(f"Total rows in table: {len(table_df)}")
table_df[["doc_id", "title"]].head()

Total rows in table: 200


,doc_id,title
0,4983,Microstructural development of human newborn cerebral white matter assessed in vivo by...
1,5836,Induction of myelodysplasia by myeloid-derived suppressor cells.
2,7912,"BC1 RNA, the transcript from a master gene for ID element amplification, is able to pr..."
3,18670,The DNA Methylome of Human Peripheral Blood Mononuclear Cells
4,19238,The human myelin basic protein gene is included within a 179-kilobase transcription un...


### 4. Fast Scalar Filtering

BenoStreamDB uses **Roaring Bitmaps** inside its Inverted Index sidecars for instant scalar lookups without scanning parquet rows.

In [5]:
# Filter papers containing 'cells' or 'cancer' in title
cellular_papers = table.filter("title LIKE '%cells%' OR title LIKE '%cancer%'").to_pandas()
print(f"Found {len(cellular_papers)} cellular / oncology papers:")
cellular_papers[["doc_id", "title"]].head()

Found 29 cellular / oncology papers:


,doc_id,title
0,5836,Induction of myelodysplasia by myeloid-derived suppressor cells.
1,92499,The journey of developing hematopoietic stem cells.
2,104130,The suture provides a niche for mesenchymal stem cells of craniofacial bones
3,106301,Cardiac neural crest cells contribute to the dormant multipotent stem cell in the mamm...
4,123859,Tracking the fate of glomerular epithelial cells in vivo using serial multiphoton imag...


### 5. Semantic Vector Search

Search for semantically similar papers given a natural language scientific hypothesis.

In [6]:
query = "Microstructural development of human cerebral cortex and white matter"
query_vec = list(model.encode(query))

# Vector search (top-5 nearest neighbors via HNSW)
results = table.filter(vector_filter=query_vec, k=5).to_pandas()
results[["doc_id", "title", "distance"]]

,doc_id,title,distance
0,4983,Microstructural development of human newborn cerebral white matter assessed in vivo by...,0.596607
1,803312,Cerebral organoids model human brain development and microcephaly,1.149886
2,739734,Stages of the pathologic process in Alzheimer disease: age categories from 1 to 100 ye...,1.350287
3,279052,"Genomic imprinting in development, growth, behavior and stem cells.",1.378462
4,313394,A Functional Neuroimaging Study of Sound Localization: Visual Cortex Activity Predicts...,1.384867


### 6. Hybrid Search (Scalar Filter + Vector Similarity)

Seamlessly combine metadata predicate filtering with vector similarity search in a single execution.

In [7]:
# Scientific papers matching 'human' in title closest to the query
hybrid_results = table.to_pandas(
    filter="title LIKE '%human%'",
    vector_filter=query_vec,
    k=3
)
hybrid_results[["doc_id", "title", "distance"]]

,doc_id,title,distance
0,4983,Microstructural development of human newborn cerebral white matter assessed in vivo by...,0.596606
1,803312,Cerebral organoids model human brain development and microcephaly,1.149886
2,19238,The human myelin basic protein gene is included within a 179-kilobase transcription un...,1.428635


### 7. Clean Teardown

In [8]:
# Clean up database resources
if os.path.exists(db_path):
    shutil.rmtree(db_path)
print("Cleaned up demo database.")

Cleaned up demo database.
